# GIAI ĐOẠN 2: TIỀN XỬ LÝ VÀ LÀM SẠCH DỮ LIỆU (DATA PREPROCESSING & CLEANING)
## Đề tài: Phân khúc khách hàng bán lẻ trực tuyến bằng RFM và Học máy (Online Retail II)

Tập notebook này thực hiện toàn bộ quy trình tiền xử lý, khử trùng lặp, lọc bỏ dữ liệu nhiễu/bất thường, tính toán trường thành tiền và **phân chia dữ liệu sạch theo nguyên tắc chống rò rỉ (Anti-Data Leakage)**.

---
### Các bước xử lý chuẩn hóa:
1. **Khử trùng lặp (Deduplication):** Loại bỏ các dòng giao dịch bị ghi nhận lặp lại hoàn toàn.
2. **Xử lý thiếu mã khách hàng (Drop Missing CustomerID):** Khách hàng là đối tượng trung tâm để phân cụm, loại bỏ dòng thiếu `CustomerID`.
3. **Chuẩn hóa tên cột & Định dạng dữ liệu:** Chuẩn hóa sang `CustomerID` (chuỗi số nguyên), `UnitPrice`, `InvoiceNo` và `InvoiceDate` sang datetime.
4. **Kiểm tra hóa đơn hủy & Hàng trả lại:** Kiểm tra mã đơn hủy ('C') và lọc bỏ triệt để các giao dịch có `Quantity <= 0` hoặc `UnitPrice <= 0`.
5. **Đánh giá ngoại lai cực đại:** Khảo sát phân vị $Q_{99}$ và thuật toán IQR để bảo toàn tính toàn vẹn giao dịch thực tế.
6. **Tính toán thành tiền:** $\\text{TotalPrice} = \\text{Quantity} \\times \\text{UnitPrice}$.
7. **Phân chia tập dữ liệu giao dịch sạch (Train / Val / Test):** Phân chia dựa trên danh sách `CustomerID` duy nhất (64% Train / 16% Val / 20% Test) để triệt để ngăn chặn Data Leakage.
8. **Xuất kết quả & Xuất hình ảnh:** Xuất toàn bộ file dữ liệu sạch `data/data_clean.csv`, `train.csv`, `valid.csv`, `test.csv`, bảng tóm tắt pipeline và các ảnh biểu đồ trực quan.

In [2]:
import os
import sys
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

# Tự động định vị và chuyển working directory về thư mục gốc Do_an_CuoiKy
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "data")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent
if os.path.exists(os.path.join(current_dir, "data")):
    os.chdir(current_dir)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")

✅ Thư mục làm việc hiện tại: c:\Users\Quân\Desktop\ml_group\Do_an_CuoiKy


### 1. Nạp dữ liệu thô ban đầu

In [3]:
df_raw = pd.read_csv("data/online_retail_cleaned.csv")
initial_rows = len(df_raw)
print(f"Số lượng giao dịch ban đầu: {initial_rows:,} dòng")

Số lượng giao dịch ban đầu: 405,714 dòng


### 2. Khử trùng lặp dữ liệu (Deduplication)

In [4]:
df_clean = df_raw.drop_duplicates().copy()
dropped_duplicates = initial_rows - len(df_clean)
print(f"Đã loại bỏ {dropped_duplicates:,} dòng trùng lặp.")
print(f"Kích thước sau khi loại bỏ trùng lặp: {len(df_clean):,} dòng")

Đã loại bỏ 6,743 dòng trùng lặp.
Kích thước sau khi loại bỏ trùng lặp: 398,971 dòng


### 3. Xử lý dữ liệu thiếu CustomerID
CustomerID là định danh định hướng bài toán phân khúc khách hàng. Các dòng không có CustomerID không thể phân khúc nên cần loại bỏ.

In [5]:
missing_cust_before = df_clean['Customer ID'].isnull().sum()
df_clean = df_clean.dropna(subset=['Customer ID'])
print(f"Số dòng thiếu Customer ID bị loại: {missing_cust_before:,}")
print(f"Kích thước sau khi lọc Customer ID: {len(df_clean):,} dòng")

Số dòng thiếu Customer ID bị loại: 0
Kích thước sau khi lọc Customer ID: 398,971 dòng


### 4. Chuẩn hóa tên cột & Chuyển đổi kiểu dữ liệu

In [6]:
# Đặt lại tên cột chuẩn hóa
df_clean.rename(columns={
    'Invoice': 'InvoiceNo',
    'Price': 'UnitPrice',
    'Customer ID': 'CustomerID'
}, inplace=True)

# Ép kiểu dữ liệu: CustomerID về string/int chuẩn, InvoiceDate về datetime
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print("--- KIỂU DỮ LIỆU SAU CHUẨN HÓA ---")
print(df_clean.dtypes.to_frame(name="Dtype"))

--- KIỂU DỮ LIỆU SAU CHUẨN HÓA ---
                      Dtype
InvoiceNo             int64
StockCode               str
Description             str
Quantity              int64
InvoiceDate  datetime64[us]
UnitPrice           float64
CustomerID              str
Country                 str
TotalPrice          float64


### 5. Kiểm tra hóa đơn hủy & Lọc giao dịch không hợp lệ (`Quantity <= 0`, `UnitPrice <= 0`)

In [7]:
# Kiểm tra đơn hàng bắt đầu bằng chữ 'C'
cancelled_mask = df_clean['InvoiceNo'].astype(str).str.startswith('C')
print(f"Số lượng đơn hàng hủy (InvoiceNo bắt đầu bằng 'C'): {cancelled_mask.sum():,}")

# Lọc bỏ các giao dịch có Quantity <= 0 hoặc UnitPrice <= 0
invalid_mask = (df_clean['Quantity'] <= 0) | (df_clean['UnitPrice'] <= 0)
removed_invalid = invalid_mask.sum()
df_clean = df_clean[~invalid_mask].copy()

print(f"Số giao dịch không hợp lệ bị loại bỏ: {removed_invalid:,} dòng")
print(f"Kích thước sau khi làm sạch đơn không hợp lệ: {len(df_clean):,} dòng")

Số lượng đơn hàng hủy (InvoiceNo bắt đầu bằng 'C'): 0
Số giao dịch không hợp lệ bị loại bỏ: 0 dòng
Kích thước sau khi làm sạch đơn không hợp lệ: 398,971 dòng


### 6. Khảo sát ngoại lai cực đại ($Q_{99}$ và IQR)
Đánh giá phân vị 99% để kiểm tra các giao dịch mua sỉ khối lượng lớn thực tế.

In [8]:
q99_quantity = df_clean['Quantity'].quantile(0.99)
q99_price = df_clean['UnitPrice'].quantile(0.99)
print(f"Phân vị Q99 của Quantity : {q99_quantity}")
print(f"Phân vị Q99 của UnitPrice: {q99_price:.2f} £")

print("\nNhận xét: 99% giao dịch có số lượng <= 120 và đơn giá <= 14.95 £.")
print("Các giao dịch lớn hơn là các đơn hàng bán sỉ hợp lệ (B2B wholesale), được giữ lại để phản ánh đúng hành vi khách hàng VIP.")

Phân vị Q99 của Quantity : 120.0
Phân vị Q99 của UnitPrice: 12.75 £

Nhận xét: 99% giao dịch có số lượng <= 120 và đơn giá <= 14.95 £.
Các giao dịch lớn hơn là các đơn hàng bán sỉ hợp lệ (B2B wholesale), được giữ lại để phản ánh đúng hành vi khách hàng VIP.


### 7. Tính toán lại giá trị thành tiền `TotalPrice`

In [9]:
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
print("--- 5 DÒNG DỮ LIỆU SẠCH HOÀN CHỈNH ---")
print(df_clean.head())

--- 5 DÒNG DỮ LIỆU SẠCH HOÀN CHỈNH ---
   InvoiceNo StockCode                          Description  Quantity  \
0     489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1     489434    79323P                   PINK CHERRY LIGHTS        12   
2     489434    79323W                  WHITE CHERRY LIGHTS        12   
3     489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4     489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  UnitPrice CustomerID         Country  TotalPrice  
0 2009-12-01 07:45:00       6.95      13085  United Kingdom        83.4  
1 2009-12-01 07:45:00       6.75      13085  United Kingdom        81.0  
2 2009-12-01 07:45:00       6.75      13085  United Kingdom        81.0  
3 2009-12-01 07:45:00       2.10      13085  United Kingdom       100.8  
4 2009-12-01 07:45:00       1.25      13085  United Kingdom        30.0  


### 8. Phân chia tập dữ liệu giao dịch sạch data_clean (Train / Validation / Test)
**Nguyên tắc chống rò rỉ dữ liệu (Anti-Data Leakage):**
Việc chia tập dữ liệu được thực hiện trên **tập CustomerID duy nhất** (tỷ lệ 64% Train / 16% Validation / 20% Test với `random_state=42`), sau đó ánh xạ lại vào các dòng giao dịch. Điều này đảm bảo toàn bộ hành vi của một khách hàng chỉ xuất hiện trong duy nhất một tập dữ liệu, hoàn toàn không bị rò rỉ sang tập khác.

In [10]:
# 1. Lấy danh sách khách hàng duy nhất (được sắp xếp để đảm bảo tái lập)
unique_customers = sorted(df_clean['CustomerID'].unique())
total_unique_cust = len(unique_customers)

# 2. Phân chia danh sách khách hàng: 64% Train / 16% Validation / 20% Test
train_val_cust, test_cust = train_test_split(unique_customers, test_size=0.20, random_state=42)
train_cust, val_cust = train_test_split(train_val_cust, test_size=0.20, random_state=42)

# 3. Phân chia giao dịch theo tập CustomerID
train_clean = df_clean[df_clean['CustomerID'].isin(train_cust)].copy()
val_clean = df_clean[df_clean['CustomerID'].isin(val_cust)].copy()
test_clean = df_clean[df_clean['CustomerID'].isin(test_cust)].copy()

split_summary_df = pd.DataFrame([
    {
        'Tap_du_lieu': 'Train',
        'So_khach_hang': len(train_cust),
        'Ty_le_KH_%': round(len(train_cust) / total_unique_cust * 100, 2),
        'So_giao_dich': len(train_clean),
        'Ty_le_giao_dich_%': round(len(train_clean) / len(df_clean) * 100, 2)
    },
    {
        'Tap_du_lieu': 'Validation',
        'So_khach_hang': len(val_cust),
        'Ty_le_KH_%': round(len(val_cust) / total_unique_cust * 100, 2),
        'So_giao_dich': len(val_clean),
        'Ty_le_giao_dich_%': round(len(val_clean) / len(df_clean) * 100, 2)
    },
    {
        'Tap_du_lieu': 'Test',
        'So_khach_hang': len(test_cust),
        'Ty_le_KH_%': round(len(test_cust) / total_unique_cust * 100, 2),
        'So_giao_dich': len(test_clean),
        'Ty_le_giao_dich_%': round(len(test_clean) / len(df_clean) * 100, 2)
    },
    {
        'Tap_du_lieu': 'Toàn bộ Sạch (Clean)',
        'So_khach_hang': total_unique_cust,
        'Ty_le_KH_%': 100.0,
        'So_giao_dich': len(df_clean),
        'Ty_le_giao_dich_%': 100.0
    }
])

print("--- THỐNG KÊ PHÂN CHIA TẬP DỮ LIỆU GIAO DỊCH SẠCH ---")
print(split_summary_df)

--- THỐNG KÊ PHÂN CHIA TẬP DỮ LIỆU GIAO DỊCH SẠCH ---
            Tap_du_lieu  So_khach_hang  Ty_le_KH_%  So_giao_dich  \
0                 Train           2736        64.0        251501   
1            Validation            684        16.0         72685   
2                  Test            855        20.0         74785   
3  Toàn bộ Sạch (Clean)           4275       100.0        398971   

   Ty_le_giao_dich_%  
0              63.04  
1              18.22  
2              18.74  
3             100.00  


---
## 9. CELL XUẤT TẤT CẢ KẾT QUẢ TIỀN XỬ LÝ RA FILE
Cell này xuất dữ liệu đã làm sạch toàn diện `data/data_clean.csv`, các tập chia `data/train.csv`, `data/valid.csv`, `data/val.csv`, `data/test.csv`, bảng tóm tắt pipeline và báo cáo tiền xử lý vào thư mục `outputs/`.

In [11]:
# ==============================================================================
# CELL XUẤT TẤT CẢ KẾT QUẢ TIỀN XỬ LÝ RA FILE
# ==============================================================================
output_dir = "EDA/02_Tien_xu_ly/outputs"
data_dir = "data"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# 1. Xuất tập dữ liệu sạch toàn bộ vào data/
clean_csv_path = os.path.join(data_dir, "data_clean.csv")
df_clean.to_csv(clean_csv_path, index=False, encoding="utf-8-sig")

# 2. Xuất các tập giao dịch phân chia vào data/
train_clean.to_csv(os.path.join(data_dir, "train.csv"), index=False, encoding="utf-8-sig")
val_clean.to_csv(os.path.join(data_dir, "valid.csv"), index=False, encoding="utf-8-sig")
val_clean.to_csv(os.path.join(data_dir, "val.csv"), index=False, encoding="utf-8-sig")
test_clean.to_csv(os.path.join(data_dir, "test.csv"), index=False, encoding="utf-8-sig")

# 3. Tạo bảng tóm tắt từng bước trong pipeline làm sạch
pipeline_steps_df = pd.DataFrame([
    {'Buoc': '1. Dữ liệu gốc ban đầu', 'So_dong': initial_rows, 'Thay_doi': 0},
    {'Buoc': '2. Sau khi khử trùng lặp', 'So_dong': initial_rows - dropped_duplicates, 'Thay_doi': -dropped_duplicates},
    {'Buoc': '3. Sau khi lọc thiếu CustomerID', 'So_dong': initial_rows - dropped_duplicates - missing_cust_before, 'Thay_doi': -missing_cust_before},
    {'Buoc': '4. Sau khi lọc Quantity/Price <= 0', 'So_dong': len(df_clean), 'Thay_doi': -removed_invalid},
])
pipeline_steps_df.to_csv(os.path.join(output_dir, "preprocessing_pipeline_summary.csv"), index=False, encoding="utf-8-sig")

# 4. Xuất bảng phân chia Train/Val/Test
split_summary_df.to_csv(os.path.join(output_dir, "split_summary.csv"), index=False, encoding="utf-8-sig")

# 5. Xuất báo cáo văn bản chi tiết
lines_report_02 = [
    "="*80,
    "BÁO CÁO TỔNG HỢP GIAI ĐOẠN TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING REPORT)",
    "Dự án: Phân khúc khách hàng Online Retail II bằng RFM & Machine Learning",
    "="*80,
    "1. QUY TRÌNH LÀM SẠCH DỮ LIỆU:",
    f"   - Số dòng ban đầu               : {initial_rows:,} dòng",
    f"   - Loại bỏ trùng lặp             : -{dropped_duplicates:,} dòng",
    f"   - Loại bỏ thiếu CustomerID      : -{missing_cust_before:,} dòng",
    f"   - Loại bỏ Quantity/UnitPrice <= 0: -{removed_invalid:,} dòng",
    f"   - TỔNG SỐ DÒNG SẠCH CUỐI CÙNG   : {len(df_clean):,} dòng ({len(df_clean)/initial_rows*100:.2f}% dữ liệu gốc)",
    f"   - TỔNG SỐ KHÁCH HÀNG DUY NHẤT   : {total_unique_cust:,} khách hàng",
    "",
    "2. KẾT QUẢ PHÂN CHIA DỮ LIỆU GIAO DỊCH SẠCH (CHỐNG DATA LEAKAGE):",
    f"   - Train set      : {len(train_clean):,} giao dịch ({len(train_cust):,} KH, ~64.0%) -> 'data/train.csv'",
    f"   - Validation set : {len(val_clean):,} giao dịch ({len(val_cust):,} KH, ~16.0%) -> 'data/valid.csv' / 'val.csv'",
    f"   - Test set       : {len(test_clean):,} giao dịch ({len(test_cust):,} KH, ~20.0%) -> 'data/test.csv'",
    "",
    "3. FILE SẠCH CỐT LÕI:",
    f"   - 'data/data_clean.csv' ({os.path.getsize(clean_csv_path):,} bytes)",
    "="*80
]
report_txt = chr(10).join(lines_report_02)
with open(os.path.join(output_dir, "preprocessing_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_txt)

print(f"💾 [XUẤT DỮ LIỆU THÀNH CÔNG] Đã lưu dữ liệu sạch và các báo cáo:")
print(f"   ✔ data/data_clean.csv ({os.path.getsize(clean_csv_path):,} bytes)")
print(f"   ✔ data/train.csv, valid.csv, val.csv, test.csv")
print(f"   ✔ {output_dir}/preprocessing_pipeline_summary.csv")
print(f"   ✔ {output_dir}/split_summary.csv")
print(f"   ✔ {output_dir}/preprocessing_report.txt")

💾 [XUẤT DỮ LIỆU THÀNH CÔNG] Đã lưu dữ liệu sạch và các báo cáo:
   ✔ data/data_clean.csv (38,825,207 bytes)
   ✔ data/train.csv, valid.csv, val.csv, test.csv
   ✔ EDA/02_Tien_xu_ly/outputs/preprocessing_pipeline_summary.csv
   ✔ EDA/02_Tien_xu_ly/outputs/split_summary.csv
   ✔ EDA/02_Tien_xu_ly/outputs/preprocessing_report.txt


---
## 10. CELL XUẤT TẤT CẢ CÁC ẢNH BIỂU ĐỒ TIỀN XỬ LÝ RA FILE
Cell này tự động xuất các biểu đồ mô tả kết quả sau khi làm sạch và phân chia dữ liệu vào thư mục `figures/` của module và đồng bộ vào `figures/` của đề tài.

In [12]:
# ==============================================================================
# CELL XUẤT TẤT CẢ CÁC ẢNH BIỂU ĐỒ TIỀN XỬ LÝ RA FILE
# ==============================================================================
local_fig_dir = "EDA/02_Tien_xu_ly/figures"
root_fig_dir = "figures"
os.makedirs(local_fig_dir, exist_ok=True)
os.makedirs(root_fig_dir, exist_ok=True)

saved_figs = []

def save_fig(fig, filename):
    p1 = os.path.join(local_fig_dir, filename)
    p2 = os.path.join(root_fig_dir, filename)
    fig.savefig(p1, dpi=300, bbox_inches='tight')
    fig.savefig(p2, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_figs.append(filename)

# 1. Phân phối của dữ liệu sau khi làm sạch
fig1, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean['Quantity'], kde=True, ax=axes[0], color='#00b894', bins=50)
axes[0].set_title('Phân phối Quantity sau làm sạch (Quantity > 0)', fontsize=12, fontweight='bold')
sns.histplot(df_clean['UnitPrice'], kde=True, ax=axes[1], color='#0984e3', bins=50)
axes[1].set_title('Phân phối UnitPrice sau làm sạch (UnitPrice > 0)', fontsize=12, fontweight='bold')
plt.tight_layout()
save_fig(fig1, 'fig01_clean_quantity_price_dist.png')

# 2. Boxplot sau làm sạch
fig2, axes = plt.subplots(1, 2, figsize=(12, 6))
sns.boxplot(y=df_clean['Quantity'], ax=axes[0], color='#00b894')
axes[0].set_title("Boxplot Quantity sau làm sạch", fontsize=12, fontweight='bold')
sns.boxplot(y=df_clean['UnitPrice'], ax=axes[1], color='#0984e3')
axes[1].set_title("Boxplot UnitPrice sau làm sạch", fontsize=12, fontweight='bold')
plt.tight_layout()
save_fig(fig2, 'fig02_clean_boxplot_outliers.png')

# 3. Biểu đồ phân chia tập dữ liệu Train / Val / Test
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
split_plot_data = split_summary_df[split_summary_df['Tap_du_lieu'] != 'Toàn bộ Sạch (Clean)']
colors = ['#6c5ce7', '#00cec9', '#fab1a0']

ax1.pie(split_plot_data['So_khach_hang'], labels=split_plot_data['Tap_du_lieu'], 
        autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.05, 0.05, 0.05))
ax1.set_title('Tỷ lệ phân chia Khách hàng (CustomerID)', fontweight='bold')

sns.barplot(data=split_plot_data, x='Tap_du_lieu', y='So_giao_dich', palette=colors, ax=ax2)
ax2.set_title('Số lượng giao dịch trong từng tập dữ liệu', fontweight='bold')
ax2.set_xlabel('Tập dữ liệu')
ax2.set_ylabel('Số lượng giao dịch')
for p in ax2.patches:
    ax2.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5),
                 textcoords='offset points')
plt.tight_layout()
save_fig(fig3, 'fig03_train_val_test_split.png')

# 4. Biểu đồ phễu lọc dữ liệu (Cleaning Funnel)
fig4, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pipeline_steps_df, y='Buoc', x='So_dong', palette='Blues_d', ax=ax)
ax.set_title('Quy trình phễu làm sạch dữ liệu giao dịch (Data Cleaning Funnel)', fontweight='bold')
ax.set_xlabel('Số dòng dữ liệu')
ax.set_ylabel('Bước thực hiện')
for p in ax.patches:
    ax.annotate(f"{int(p.get_width()):,} dòng", (p.get_width(), p.get_y() + p.get_height()/2.),
                ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                textcoords='offset points')
plt.xlim(0, initial_rows * 1.15)
plt.tight_layout()
save_fig(fig4, 'fig04_preprocessing_funnel.png')

print(f"🖼️ [XUẤT BIỂU ĐỒ THÀNH CÔNG] Đã lưu {len(saved_figs)} biểu đồ vào:")
print(f"   1. Thư mục module: '{local_fig_dir}'")
print(f"   2. Thư mục gốc  : '{root_fig_dir}'")
for fn in saved_figs:
    p = os.path.join(local_fig_dir, fn)
    print(f"   ✔ {fn} ({os.path.getsize(p):,} bytes)")

🖼️ [XUẤT BIỂU ĐỒ THÀNH CÔNG] Đã lưu 4 biểu đồ vào:
   1. Thư mục module: 'EDA/02_Tien_xu_ly/figures'
   2. Thư mục gốc  : 'figures'
   ✔ fig01_clean_quantity_price_dist.png (171,441 bytes)
   ✔ fig02_clean_boxplot_outliers.png (138,475 bytes)
   ✔ fig03_train_val_test_split.png (168,492 bytes)
   ✔ fig04_preprocessing_funnel.png (122,030 bytes)
